# ElectroNest — Free Deployment Guide

Deploy your full-stack app (Django backend + React frontend + SQL Server data) for free so it works like a real website.

---

## Architecture Overview

```
Browser → Vercel (React frontend) → Railway (Django API) → Railway (PostgreSQL DB)
```

> **Why not SQL Server?** Free cloud hosting (Railway, Render, Supabase) does not support SQL Server. You will migrate your data to **PostgreSQL** (free). The Django ORM makes this straightforward.

---

## Step 1 — Migrate Database from SQL Server to PostgreSQL

### 1.1 Export your current data

Run this in your Django project while connected to SQL Server:

```bash
cd backend
python manage.py dumpdata --natural-foreign --natural-primary \
  --exclude auth.permission --exclude contenttypes \
  -o data_backup.json
```

> If `dumpdata` fails on SQL Server with `managed = False` models, use SQL Server Management Studio (SSMS) to export each table as CSV, then import manually after setting up PostgreSQL.

### 1.2 Switch Django to PostgreSQL

Install the PostgreSQL driver:

```bash
pip install psycopg2-binary
pip freeze > requirements.txt
```

Update `backend/page/settings.py`:

```python
import os

DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.postgresql',
        'NAME': os.environ.get('DB_NAME', 'electronest'),
        'USER': os.environ.get('DB_USER', 'postgres'),
        'PASSWORD': os.environ.get('DB_PASSWORD', ''),
        'HOST': os.environ.get('DB_HOST', 'localhost'),
        'PORT': os.environ.get('DB_PORT', '5432'),
    }
}
```

Also update these settings for production:

```python
import os

SECRET_KEY = os.environ.get('SECRET_KEY', 'your-dev-key-here')
DEBUG = os.environ.get('DEBUG', 'False') == 'True'
ALLOWED_HOSTS = os.environ.get('ALLOWED_HOSTS', 'localhost').split(',')

# CORS — allow your Vercel frontend
CORS_ALLOWED_ORIGINS = os.environ.get('CORS_ORIGINS', 'http://localhost:5173').split(',')
```

### 1.3 Fix managed = False models

Your Review, Product, etc. models use `managed = False`. For PostgreSQL you need Django to create the tables. Temporarily change each model:

```python
class Meta:
    db_table = 'Reviews'
    managed = True          # change to True for migration
```

Run migrations:

```bash
python manage.py makemigrations
python manage.py migrate
```

Then restore your data:

```bash
python manage.py loaddata data_backup.json
```

After confirming data loaded, you can set `managed = False` back (or leave as `True`).

---

## Step 2 — Deploy Backend on Railway (Free)

Railway gives you a free PostgreSQL database + Python hosting.

### 2.1 Create a Railway account

Go to [railway.app](https://railway.app) and sign up with GitHub.

### 2.2 Add a PostgreSQL database

1. Click **New Project** → **Deploy PostgreSQL**
2. After it provisions, click on the database → **Connect** tab
3. Copy the connection variables: `PGHOST`, `PGPORT`, `PGDATABASE`, `PGUSER`, `PGPASSWORD`

### 2.3 Push your backend to GitHub

```bash
cd backend
git init
git add .
git commit -m "Initial backend commit"
# Create a new repo on GitHub, then:
git remote add origin https://github.com/YOUR_USERNAME/electronest-backend.git
git push -u origin main
```

### 2.4 Deploy Django on Railway

1. In Railway → **New Service** → **GitHub Repo** → select your backend repo
2. Set the **Root Directory** to `backend` (if your repo root is the whole project)
3. Add these **Environment Variables** in Railway dashboard:

| Variable | Value |
|---|---|
| `SECRET_KEY` | any long random string |
| `DEBUG` | `False` |
| `ALLOWED_HOSTS` | `your-app.railway.app` |
| `CORS_ORIGINS` | `https://your-app.vercel.app` |
| `DB_NAME` | from Railway PostgreSQL |
| `DB_USER` | from Railway PostgreSQL |
| `DB_PASSWORD` | from Railway PostgreSQL |
| `DB_HOST` | from Railway PostgreSQL |
| `DB_PORT` | `5432` |

4. Add a `Procfile` in your backend folder:

```
web: gunicorn page.wsgi --bind 0.0.0.0:$PORT
```

5. Install gunicorn:

```bash
pip install gunicorn
pip freeze > requirements.txt
```

6. Railway will auto-detect `requirements.txt` and deploy. Wait for the build to finish.

7. Run migrations on Railway via the **Shell** tab:

```bash
python manage.py migrate
python manage.py loaddata data_backup.json
python manage.py collectstatic --noinput
```

8. Note your Railway backend URL, e.g. `https://electronest-backend.up.railway.app`

---

## Step 3 — Deploy Frontend on Vercel (Free)

### 3.1 Update the API base URL

In `frontend/src/services/api.js`, change the base URL to use an environment variable:

```js
const BASE_URL = import.meta.env.VITE_API_URL || 'http://localhost:8000/api';
```

Create a file `frontend/.env.production`:

```
VITE_API_URL=https://electronest-backend.up.railway.app/api
```

> Do NOT commit `.env.production` with secrets — set variables in Vercel dashboard instead.

### 3.2 Push frontend to GitHub

```bash
cd frontend
git init
git add .
git commit -m "Initial frontend commit"
git remote add origin https://github.com/YOUR_USERNAME/electronest-frontend.git
git push -u origin main
```

### 3.3 Deploy on Vercel

1. Go to [vercel.com](https://vercel.com) and sign up with GitHub
2. Click **Add New Project** → import your frontend repo
3. Set **Framework Preset** to `Vite`
4. Set **Root Directory** to `frontend` (if needed)
5. Add **Environment Variables**:
   - `VITE_API_URL` = `https://electronest-backend.up.railway.app/api`
6. Click **Deploy**

Vercel gives you a live URL like `https://electronest.vercel.app`

---

## Step 4 — Serve Django Media Files (Product Images)

By default, Django serves media files locally. For production, use **Cloudinary** (free tier):

```bash
pip install cloudinary django-cloudinary-storage
pip freeze > requirements.txt
```

Sign up at [cloudinary.com](https://cloudinary.com) and add to `settings.py`:

```python
import cloudinary
import cloudinary.uploader
import cloudinary.api

INSTALLED_APPS += ['cloudinary_storage', 'cloudinary']

DEFAULT_FILE_STORAGE = 'cloudinary_storage.storage.MediaCloudinaryStorage'

CLOUDINARY_STORAGE = {
    'CLOUD_NAME': os.environ.get('CLOUDINARY_CLOUD_NAME'),
    'API_KEY': os.environ.get('CLOUDINARY_API_KEY'),
    'API_SECRET': os.environ.get('CLOUDINARY_API_SECRET'),
}
```

Add these 3 variables to your Railway environment variables.

Then re-upload your product images — any new image uploaded through Django admin or API will automatically go to Cloudinary.

---

## Step 5 — Final Checklist

- [ ] Backend URL responds: `https://your-backend.up.railway.app/api/products/`
- [ ] Frontend loads: `https://your-app.vercel.app`
- [ ] Products display with images
- [ ] Login / register works
- [ ] Add to cart, checkout works
- [ ] Admin panel accessible at `/api/admin/` (Django admin)
- [ ] CORS is configured to allow your Vercel domain

---

## Free Tier Limits

| Service | Free Limit |
|---|---|
| Railway | $5 credit/month (enough for a small app), 500 MB PostgreSQL |
| Vercel | Unlimited frontend deploys, 100 GB bandwidth/month |
| Cloudinary | 25 GB storage, 25 GB bandwidth/month |

---

## Quick Reference — Deployment Commands

```bash
# Backend (run once after deploy)
python manage.py migrate
python manage.py loaddata data_backup.json
python manage.py createsuperuser      # create admin account
python manage.py collectstatic --noinput

# Local dev (unchanged)
cd backend && python manage.py runserver
cd frontend && npm run dev
```

---

## Troubleshooting

**CORS error in browser** — Make sure `CORS_ALLOWED_ORIGINS` in Railway env variables includes your exact Vercel URL (no trailing slash).

**500 error on Railway** — Check Railway logs. Most common cause: missing environment variable or failed migration.

**Images not loading** — Either set up Cloudinary (Step 4) or set `MEDIA_URL` to an absolute URL pointing to your Railway server and ensure `whitenoise` is serving static/media files.

**`managed = False` errors on migrate** — Set `managed = True` in all models temporarily, run migrate, then revert if needed.


# ElectroNest — Free Deployment Guide

Deploy your full-stack app (Django backend + React frontend + your database data) for free so it works like a real website.

Your project is already on GitHub — this guide starts directly from connecting it to the hosting platforms.

---

## Architecture

```
Browser → Vercel (React frontend) → Railway (Django API) → Railway (PostgreSQL DB)
```

> **Why PostgreSQL instead of SQL Server?**
> Free cloud hosts (Railway, Render) don't support SQL Server / ODBC.
> You export your data once, switch Django to PostgreSQL, and import it back. The ORM handles the rest.

---

## Step 1 — Migrate SQL Server → PostgreSQL locally

### 1.1 Export your existing data

With your SQL Server still running:

```bash
cd backend
python manage.py dumpdata --natural-foreign --natural-primary \
  --exclude auth.permission --exclude contenttypes \
  -o data_backup.json
```

> If any model has `managed = False`, set it to `managed = True` temporarily before running this.

### 1.2 Install PostgreSQL driver

```bash
pip install psycopg2-binary gunicorn
pip freeze > requirements.txt
```

### 1.3 Update `backend/page/settings.py`

Replace the entire `DATABASES` block and add environment-variable support:

```python
import os

SECRET_KEY = os.environ.get('SECRET_KEY', 'your-local-dev-key')
DEBUG = os.environ.get('DEBUG', 'False') == 'True'
ALLOWED_HOSTS = os.environ.get('ALLOWED_HOSTS', 'localhost').split(',')

DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.postgresql',
        'NAME':     os.environ.get('DB_NAME', 'electronest'),
        'USER':     os.environ.get('DB_USER', 'postgres'),
        'PASSWORD': os.environ.get('DB_PASSWORD', ''),
        'HOST':     os.environ.get('DB_HOST', 'localhost'),
        'PORT':     os.environ.get('DB_PORT', '5432'),
    }
}

# CORS — allow your Vercel frontend
CORS_ALLOWED_ORIGINS = os.environ.get('CORS_ORIGINS', 'http://localhost:5173').split(',')
```

### 1.4 Set all models to `managed = True`

In `backend/products/models.py` (and any other app with `managed = False`):

```python
class Meta:
    db_table = 'Reviews'
    managed = True   # was False
```

### 1.5 Run migrations and restore data

```bash
python manage.py makemigrations
python manage.py migrate
python manage.py loaddata data_backup.json
```

### 1.6 Add a `Procfile` in the `backend/` folder

```
web: gunicorn page.wsgi --bind 0.0.0.0:$PORT
```

### 1.7 Commit and push these changes

```bash
git add backend/
git commit -m "Switch to PostgreSQL and add env-var config for deployment"
git push
```

---

## Step 2 — Deploy Backend + Database on Railway

### 2.1 Create account & project

1. Go to [railway.app](https://railway.app) → sign in with GitHub
2. Click **New Project** → **Deploy from GitHub repo**
3. Select your repository
4. Set **Root Directory** to `backend`

### 2.2 Add a PostgreSQL database

Inside the same project on Railway:
1. Click **+ New** → **Database** → **Add PostgreSQL**
2. Railway automatically creates it and links the credentials

### 2.3 Set environment variables

In your backend service → **Variables** tab, add:

| Variable | Value |
|---|---|
| `SECRET_KEY` | any long random string (e.g. from [djecrety.ir](https://djecrety.ir)) |
| `DEBUG` | `False` |
| `ALLOWED_HOSTS` | `your-app.up.railway.app` (Railway gives you this URL after deploy) |
| `CORS_ORIGINS` | `https://your-app.vercel.app` (fill in after Vercel deploy) |
| `DB_NAME` | copy from Railway PostgreSQL → **Connect** tab |
| `DB_USER` | copy from Railway PostgreSQL → **Connect** tab |
| `DB_PASSWORD` | copy from Railway PostgreSQL → **Connect** tab |
| `DB_HOST` | copy from Railway PostgreSQL → **Connect** tab |
| `DB_PORT` | `5432` |

### 2.4 Run migrations on Railway

Once deployed, go to your backend service → **Shell** tab:

```bash
python manage.py migrate
python manage.py loaddata data_backup.json
python manage.py createsuperuser
python manage.py collectstatic --noinput
```

Your backend is now live at: `https://your-app.up.railway.app`

Test it: open `https://your-app.up.railway.app/api/products/` — you should see your products as JSON.

---

## Step 3 — Update Frontend API URL

### 3.1 Edit `frontend/src/services/api.js`

Find the base URL line and change it to use an environment variable:

```js
const BASE_URL = import.meta.env.VITE_API_URL || 'http://localhost:8000/api';
```

### 3.2 Commit and push

```bash
git add frontend/src/services/api.js
git commit -m "Use VITE_API_URL env var for API base URL"
git push
```

---

## Step 4 — Deploy Frontend on Vercel

### 4.1 Create account

Go to [vercel.com](https://vercel.com) → sign in with GitHub

### 4.2 Import your project

1. Click **Add New Project** → select your GitHub repository
2. Set **Framework Preset** → `Vite`
3. Set **Root Directory** → `frontend`
4. Under **Environment Variables**, add:
   - `VITE_API_URL` = `https://your-app.up.railway.app/api`
5. Click **Deploy**

Your frontend is now live at: `https://your-app.vercel.app`

### 4.3 Update CORS on Railway

Go back to Railway → backend service → **Variables** tab and update:

```
CORS_ORIGINS = https://your-app.vercel.app
```

Railway will redeploy automatically.

---

## Step 5 — Product Images (Optional but Recommended)

Django's local `MEDIA_ROOT` doesn't work on Railway. Use **Cloudinary** (free):

```bash
pip install cloudinary django-cloudinary-storage
pip freeze > requirements.txt
```

Sign up at [cloudinary.com](https://cloudinary.com) → get your cloud name, API key, API secret.

Add to `settings.py`:

```python
INSTALLED_APPS += ['cloudinary_storage', 'cloudinary']

DEFAULT_FILE_STORAGE = 'cloudinary_storage.storage.MediaCloudinaryStorage'

CLOUDINARY_STORAGE = {
    'CLOUD_NAME': os.environ.get('CLOUDINARY_CLOUD_NAME'),
    'API_KEY':    os.environ.get('CLOUDINARY_API_KEY'),
    'API_SECRET': os.environ.get('CLOUDINARY_API_SECRET'),
}
```

Add the 3 Cloudinary variables to Railway environment variables, commit, and push.

---

## Final Checklist

- [ ] `https://your-backend.up.railway.app/api/products/` returns JSON with your products
- [ ] `https://your-app.vercel.app` loads the homepage with products
- [ ] Login / register works
- [ ] Add to cart and checkout works
- [ ] Django admin accessible at `https://your-backend.up.railway.app/admin/`
- [ ] No CORS errors in browser console

---

## Free Tier Limits

| Service | Free Allowance |
|---|---|
| Railway | $5 credit/month — enough for backend + PostgreSQL on a small app |
| Vercel | Unlimited deploys, 100 GB bandwidth/month |
| Cloudinary | 25 GB storage + 25 GB bandwidth/month |

---

## Troubleshooting

**CORS error in browser** — `CORS_ORIGINS` in Railway must exactly match your Vercel URL, no trailing slash.

**500 on Railway** — Check Railway **Logs** tab. Usually a missing env variable or failed migration.

**Images not showing** — Set up Cloudinary (Step 5) or check that `MEDIA_URL` is set correctly.

**`loaddata` fails** — Try loading app by app: `python manage.py loaddata data_backup.json --app products`
